***Meg-Go task***

With MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-go(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   env_kurt_R_mean  env_kurt_R_std  env_kurt_L_mean  env_kurt_L_std  \
0         5.749049       15.894836         5.962378       17.165772   
1         4.459546        7.184608         4.390848        6.578943   
2         1.161476        0.718562         1.885349        1.714298   
3         1.138575        0.699601         1.365950        0.817770   
4     18283.789736     4480.567058     18277.357293     4487.602704   

   burst_dur_cv_L_std  env_mod_rate_L_std  env_skew_L_mean  PD_Label_Combined  
0        4.938551e-02            0.244843         1.123760                  0  
1        1.147524e-01            0.117228         1.400286                  0  
2        5.040195e-02            0.275503         0.953293                  0  
3        4.311462e-02            0.460356         0.920828                  1  
4        1.043683e-16            0.064449       129.243639                  1  


In [3]:
# ============================================================
#     SIMPLE + STABLE DANN + SCL + MLP CLASSIFIER
#     CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function
import torch.nn.functional as F

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LOAD DATA
# ============================================================

# First 7 columns -> Features
# Last column     -> PD / HC labels


# ============================================================
# MEG FEATURES AND LABELS
# ============================================================

# All columns except the last one are MEG features
X_meg = final_meg_df.iloc[:, :-1].copy()

# Convert features to numeric
X_meg = X_meg.apply(
    pd.to_numeric,
    errors="coerce"
)

# Replace +inf and -inf with NaN
X_meg = X_meg.replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------
# Check invalid values
# ------------------------------------------------------------

print(
    "NaN values before cleaning:",
    X_meg.isna().sum().sum()
)

print(
    "Infinite values before cleaning:",
    np.isinf(
        X_meg.select_dtypes(include=np.number)
    ).sum().sum()
)

# ------------------------------------------------------------
# Replace NaN with column median
# ------------------------------------------------------------

X_meg = X_meg.fillna(
    X_meg.median(numeric_only=True)
)

# Any remaining NaN -> 0
X_meg = X_meg.fillna(0)

# Convert to float64 to avoid float32 overflow
X_meg = X_meg.values.astype(np.float64)

# Labels
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int64)

# ============================================================
# FINAL CHECK BEFORE STANDARDIZATION
# ============================================================

print(
    "All MEG values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Maximum absolute MEG value:",
    np.max(np.abs(X_meg))
)

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_meg = StandardScaler()

X_meg = scaler_meg.fit_transform(
    X_meg
)

# ============================================================
# FINAL CHECK AFTER STANDARDIZATION
# ============================================================

print(
    "All standardized values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Standardized MEG shape:",
    X_meg.shape
)

# ============================================================
# DOMAIN LABELS
# ============================================================

# 0 -> Acoustic
# 1 -> MEG

domain_meg = np.ones(
    len(X_meg)
)

# ============================================================
# COMBINE
# ============================================================

X_all = np.vstack([

    X_meg
]).astype(np.float32)

task_labels = np.hstack([

    y_meg
]).astype(np.int64)

domain_labels = np.hstack([

    domain_meg
]).astype(np.int64)

# ============================================================
# TENSORS
# ============================================================

X_all_tensor = torch.tensor(
    X_all
).float().to(device)

task_labels_tensor = torch.tensor(
    task_labels
).long().to(device)

domain_labels_tensor = torch.tensor(
    domain_labels
).long().to(device)

# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = lambd

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):

    return GradReverse.apply(x, lambd)

# ============================================================
# SIMPLE RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),
            nn.ReLU(),

            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )

# ============================================================
# DANN + SCL MODEL
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # ====================================================
        # SIMPLE ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        # ====================================================
        # TASK HEAD
        # ====================================================

        self.task_head = nn.Sequential(

            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        # ====================================================
        # DOMAIN HEAD
        # ====================================================

        self.domain_head = nn.Sequential(

            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        # L2 NORMALIZATION

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        # DANN

        rev_z = grad_reverse(
            z_norm,
            lambd
        )

        task_out = self.task_head(z_norm)

        domain_out = self.domain_head(rev_z)

        return task_out, domain_out, z_norm

# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        device = features.device

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(
            mask.shape[0]
        ).to(device)

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(1, keepdim=True) + 1e-8
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(1) / (mask.sum(1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# ============================================================
# DATA LOADER
# ============================================================

dataset = TensorDataset(
    X_all_tensor,
    task_labels_tensor,
    domain_labels_tensor
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = DANNContrastive(
    input_dim=X_all.shape[1],
    latent_dim=32
).to(device)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LOSSES
# ============================================================

task_loss_fn = nn.CrossEntropyLoss()

domain_loss_fn = nn.CrossEntropyLoss()

contrastive_loss_fn = SupConLoss(
    temperature=0.07
)

epochs = 120

print("\nTraining Started...\n")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(epochs):

    model.train()

    total_correct = 0
    total_samples = 0

    # ========================================================
    # WEAKER DANN
    # ========================================================

    p = min(epoch / 50.0, 1.0)

    lambd = 0.25 * (
        2 / (1 + np.exp(-10 * p)) - 1
    )

    for xb, y_task, y_dom in loader:

        optimizer.zero_grad()

        task_out, dom_out, z = model(
            xb,
            lambd
        )

        # ====================================================
        # LOSSES
        # ====================================================

        loss_task = task_loss_fn(
            task_out,
            y_task
        )

        loss_domain = domain_loss_fn(
            dom_out,
            y_dom
        )

        loss_contrastive = contrastive_loss_fn(
            z,
            y_task
        )

        # ====================================================
        # FINAL LOSS
        # ====================================================

        loss = (
            3.0 * loss_task +
            0.25 * loss_domain +
            0.55 * loss_contrastive
        )

        loss.backward()

        optimizer.step()

        preds = task_out.argmax(1)

        total_correct += (
            preds == y_task
        ).sum().item()

        total_samples += len(y_task)

    acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| λ={lambd:.3f} "
        f"| Acc={acc:.4f}"
    )

# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

model.eval()

with torch.no_grad():

    fused_meg_32D = model(
        torch.tensor(X_meg).float().to(device),
        0
    )[2].cpu().numpy()

# ============================================================
# COMBINE LATENTS
# ============================================================

X_fused = np.vstack([

    fused_meg_32D
])

y_fused = np.hstack([

    y_meg
])

# ============================================================
# SIMPLE MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# CROSS PARTICIPANT EVALUATION
# ============================================================

print("\n======================================")
print(" CROSS PARTICIPANT ANALYSIS")
print("======================================\n")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accs = []
precs = []
recs = []
f1s = []
aucs = []

fold = 1

for train_idx, test_idx in skf.split(X_fused, y_fused):

    X_train = torch.tensor(
        X_fused[train_idx]
    ).float().to(device)

    X_test = torch.tensor(
        X_fused[test_idx]
    ).float().to(device)

    y_train = torch.tensor(
        y_fused[train_idx]
    ).long().to(device)

    y_test = y_fused[test_idx]

    clf = FinalMLP().to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # TRAIN MLP
    # ========================================================

    clf.train()

    for epoch in range(80):

        optimizer_clf.zero_grad()

        outputs = clf(X_train)

        loss = criterion(
            outputs,
            y_train
        )

        loss.backward()

        optimizer_clf.step()

    # ========================================================
    # TEST
    # ========================================================

    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, preds)

    prec = precision_score(y_test, preds)

    rec = recall_score(y_test, preds)

    f1 = f1_score(y_test, preds)

    auc = roc_auc_score(y_test, probs)

    accs.append(acc)
    precs.append(prec)
    recs.append(rec)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold}")

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    print()

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print(" FINAL RESULTS")
print("======================================\n")

print(f"Mean Accuracy  : {np.mean(accs)*100:.2f}%")
print(f"Mean Precision : {np.mean(precs)*100:.2f}%")
print(f"Mean Recall    : {np.mean(recs)*100:.2f}%")
print(f"Mean F1-Score  : {np.mean(f1s)*100:.2f}%")
print(f"Mean AUC       : {np.mean(aucs)*100:.2f}%")

print("\n======================================")
print(" LATENT SHAPES")
print("======================================\n")


print("MEG      :", fused_meg_32D.shape)

NaN values before cleaning: 0
Infinite values before cleaning: 0
All MEG values finite: True
Maximum absolute MEG value: 18283.789736342947
All standardized values finite: True
Standardized MEG shape: (127, 7)

Training Started...

Epoch 1/120 | λ=0.000 | Acc=0.4803
Epoch 2/120 | λ=0.025 | Acc=0.4803
Epoch 3/120 | λ=0.049 | Acc=0.4724
Epoch 4/120 | λ=0.073 | Acc=0.4803
Epoch 5/120 | λ=0.095 | Acc=0.4488
Epoch 6/120 | λ=0.116 | Acc=0.5276
Epoch 7/120 | λ=0.134 | Acc=0.4331
Epoch 8/120 | λ=0.151 | Acc=0.4409
Epoch 9/120 | λ=0.166 | Acc=0.5276
Epoch 10/120 | λ=0.179 | Acc=0.5118
Epoch 11/120 | λ=0.190 | Acc=0.5669
Epoch 12/120 | λ=0.200 | Acc=0.4803
Epoch 13/120 | λ=0.208 | Acc=0.5118
Epoch 14/120 | λ=0.215 | Acc=0.5276
Epoch 15/120 | λ=0.221 | Acc=0.4961
Epoch 16/120 | λ=0.226 | Acc=0.4567
Epoch 17/120 | λ=0.230 | Acc=0.5669
Epoch 18/120 | λ=0.234 | Acc=0.5433
Epoch 19/120 | λ=0.237 | Acc=0.5433
Epoch 20/120 | λ=0.239 | Acc=0.5039
Epoch 21/120 | λ=0.241 | Acc=0.5512
Epoch 22/120 | λ=0.24

In [4]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)

fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([127, 32])
torch.float32


In [5]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)


# MEG target from final_meg_df (last column)
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
 # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

# (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target

meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([127, 32]), dtype: torch.float32
Labels shape: torch.Size([127]), dtype: torch.int64


In [6]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(127, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0 -0.159626  0.184386  0.200312 -0.068938  0.074314 -0.135483  0.018171   
1 -0.100833  0.223671  0.260760  0.057621  0.084586 -0.109696 -0.019146   
2 -0.107040  0.214937  0.255332  0.015525  0.099971 -0.099356 -0.027358   
3 -0.033045  0.221839  0.278378  0.140576  0.157581 -0.077821 -0.076543   
4 -0.045565  0.244998  0.284265  0.175429  0.141997 -0.052212 -0.068661   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0 -0.066970 -0.149267  0.063902  ...  0.344495 -0.128585 -0.098814  0.268364   
1 -0.047351 -0.077014 -0.012842  ...  0.388377 -0.060324 -0.119817  0.171608   
2 -0.027929 -0.064208 -0.040126  ...  0.394874 -0.064261 -0.122278  0.185366   
3  0.020569  0.051520 -0.106886  ...  0.400223  0.015430 -0.130758  0.100185   
4  0.028054  0.062387 -0.202213  ...  0.394271  0.043650 -0.111662  0.046297   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [7]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd

fused_df_named_int.to_csv('/content/drive//MyDrive/Modality-top7-par-meg(go)_MIRA-Net_data.csv', index=False)

In [10]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int[
    [f'feat_{i}' for i in range(32)]
].copy()

y = fused_df_named_int[
    "PD_Label_Combined"
].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 32)
Test Shape  : (26, 32)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 0.5455
Precision           : 0.5000
Sensitivity         : 0.8000
F1 Score            : 0.6154
ROC-AUC             : 0.7333
Balanced Accuracy   : 0.5667

XGBoost | Fold 2/10
Accuracy            : 0.9000
Precision           : 0.8000
Sensitivity         : 1.0000
F1 Score            : 0.8889
ROC-AUC            

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
Random Forest,0.682727,0.721667,0.62,0.636111,0.728500,0.685000,0.730769,0.539170,0.862956,0.800000,...,0.577654,0.956742,0.695652,0.840237,0.730769,0.474342,11.0,2.0,5.0,8.0
Stacking Fusion,0.653636,0.654286,0.68,0.648838,0.764000,0.660000,0.692308,0.500117,0.834987,0.692308,...,0.423693,0.873193,0.692308,0.804734,0.692308,0.384615,9.0,4.0,4.0,9.0
XGBoost,0.654545,0.643333,0.62,0.610803,0.706833,0.658333,0.692308,0.500117,0.834987,0.777778,...,0.577654,0.956742,0.636364,0.769231,0.692308,0.404226,11.0,2.0,6.0,7.0
Voting Fusion,0.644545,0.673333,0.60,0.595564,0.718167,0.648333,0.692308,0.500117,0.834987,0.777778,...,0.577654,0.956742,0.636364,0.781065,0.692308,0.404226,11.0,2.0,6.0,7.0
HistGradientBoosting,0.674545,0.699762,0.64,0.625059,0.726333,0.678333,0.576923,0.389486,0.744556,0.583333,...,0.355229,0.822903,0.560000,0.710059,0.576923,0.154303,8.0,5.0,6.0,7.0


Without MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-go(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   env_kurt_R_mean  env_kurt_R_std  env_kurt_L_mean  env_kurt_L_std  \
0         5.749049       15.894836         5.962378       17.165772   
1         4.459546        7.184608         4.390848        6.578943   
2         1.161476        0.718562         1.885349        1.714298   
3         1.138575        0.699601         1.365950        0.817770   
4     18283.789736     4480.567058     18277.357293     4487.602704   

   burst_dur_cv_L_std  env_mod_rate_L_std  env_skew_L_mean  PD_Label_Combined  
0        4.938551e-02            0.244843         1.123760                  0  
1        1.147524e-01            0.117228         1.400286                  0  
2        5.040195e-02            0.275503         0.953293                  0  
3        4.311462e-02            0.460356         0.920828                  1  
4        1.043683e-16            0.064449       129.243639                  1  


In [3]:
fused_df_named_int = final_meg_df.rename(columns={"Pd_indicator_meg": "PD_Label_Combined"})
fused_df_named_int.head()
fused_df_named_int.shape

(127, 8)

In [4]:
fused_df_named_int.head()

,env_kurt_R_mean,env_kurt_R_std,env_kurt_L_mean,env_kurt_L_std,burst_dur_cv_L_std,env_mod_rate_L_std,env_skew_L_mean,PD_Label_Combined
0,5.749049,15.894836,5.962378,17.165772,4.938551e-02,0.244843,1.123760,0
1,4.459546,7.184608,4.390848,6.578943,1.147524e-01,0.117228,1.400286,0
2,1.161476,0.718562,1.885349,1.714298,5.040195e-02,0.275503,0.953293,0
3,1.138575,0.699601,1.365950,0.817770,4.311462e-02,0.460356,0.920828,1
4,18283.789736,4480.567058,18277.357293,4487.602704,1.043683e-16,0.064449,129.243639,1


In [5]:
# ============================================================
# ###8)###
#
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

# All feature columns except the final label column
X = fused_df_named_int.iloc[:, :-1].copy()

# Target
y = fused_df_named_int["PD_Label_Combined"].copy()


# ============================================================
# CONVERT FEATURES TO NUMERIC
# ============================================================

X = X.apply(
    pd.to_numeric,
    errors="coerce"
)


# ============================================================
# CHECK RAW VALUES
# ============================================================

print("\nRaw feature statistics:")
print("Shape :", X.shape)

print(
    "NaN values before cleaning :",
    X.isna().sum().sum()
)

print(
    "Inf values before cleaning :",
    np.isinf(
        X.to_numpy(dtype=np.float64)
    ).sum()
)


# ============================================================
# HANDLE +INF / -INF
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# COLUMN-WISE MEDIAN IMPUTATION
# ============================================================

col_medians = X.median(
    axis=0,
    skipna=True
)

# If an entire column is NaN,
# replace its median with 0
col_medians = col_medians.fillna(0)

X = X.fillna(
    col_medians
)


# ============================================================
# CLIP EXTREMELY LARGE VALUES
# ============================================================

# Prevent overflow when converting to float32.
#
# Values larger than this are clipped rather than allowed
# to produce inf during float32 conversion.

CLIP_VALUE = 1e10

X = X.clip(
    lower=-CLIP_VALUE,
    upper=CLIP_VALUE
)


# ============================================================
# CONVERT TO FLOAT32 AFTER CLEANING
# ============================================================

X = X.astype(
    np.float32
)


# ============================================================
# FINAL FINITE-VALUE CHECK
# ============================================================

X_array = X.to_numpy()

print("\nAfter cleaning:")

print(
    "NaN values :",
    np.isnan(X_array).sum()
)

print(
    "Inf values :",
    np.isinf(X_array).sum()
)

print(
    "Finite     :",
    np.isfinite(X_array).all()
)

print(
    "Maximum    :",
    np.max(X_array)
)

print(
    "Minimum    :",
    np.min(X_array)
)


# ============================================================
# TARGET CLEANING
# ============================================================

y = pd.to_numeric(
    y,
    errors="coerce"
)

# Make sure target contains no missing values
valid_idx = y.notna()

X = X.loc[valid_idx].reset_index(drop=True)

y = y.loc[valid_idx].astype(
    np.int64
).reset_index(drop=True)


print("\nFinal data:")
print("X shape :", X.shape)
print("y shape :", y.shape)

print("\nClass distribution:")
print(y.value_counts())


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)


print("\nTraining Class Distribution:")
print(
    y_train.value_counts()
)


print("\nTesting Class Distribution:")
print(
    y_test.value_counts()
)


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw_dict[1] / cw_dict[0]
)


print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)




# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        scaler = StandardScaler()
        X_tr = X_tr.astype(np.float64)
        X_val = X_val.astype(np.float64)
        X_tr = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        # ----------------------------------------------------
        # Safety check after scaling
        # ----------------------------------------------------
        X_tr_scaled = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        print(
            f"Fold {fold}: "
            f"max|X_val|={np.max(np.abs(X_val_scaled)):.4f}"
        )

        val_preds = model.predict(
             X_val_scaled
        )
        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]

        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00

Raw feature statistics:
Shape : (127, 7)
NaN values before cleaning : 0
Inf values before cleaning : 0

After cleaning:
NaN values : 0
Inf values : 0
Finite     : True
Maximum    : 18283.79
Minimum    : 0.0

Final data:
X shape : (127, 7)
y shape : (127,)

Class distribution:
PD_Label_Combined
0    65
1    62
Name: count, dtype: int64

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 7)
Test Shape  : (26, 7)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
HistGradientBoosting,0.655455,0.657619,0.640,0.635556,0.730500,0.656667,0.730769,0.539170,0.862956,0.714286,...,0.423693,0.873193,0.740741,0.816568,0.730769,0.462910,9.0,4.0,3.0,10.0
XGBoost,0.645455,0.670833,0.680,0.645357,0.737000,0.653333,0.692308,0.500117,0.834987,0.666667,...,0.355229,0.822903,0.714286,0.798817,0.692308,0.389249,8.0,5.0,3.0,10.0
Voting Fusion,0.655455,0.684762,0.680,0.652727,0.745833,0.661667,0.692308,0.500117,0.834987,0.666667,...,0.355229,0.822903,0.714286,0.822485,0.692308,0.389249,8.0,5.0,3.0,10.0
Random Forest,0.673636,0.717143,0.615,0.643939,0.785667,0.675833,0.692308,0.500117,0.834987,0.692308,...,0.423693,0.873193,0.692308,0.763314,0.692308,0.384615,9.0,4.0,4.0,9.0
Stacking Fusion,0.623636,0.612143,0.640,0.608788,0.718000,0.631667,0.653846,0.462206,0.805878,0.642857,...,0.355229,0.822903,0.666667,0.721893,0.653846,0.308607,8.0,5.0,4.0,9.0


***Meg-passive task***

With MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-passive(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   env_kurt_asym  burst_amp_cv_asym  env_kurt_R_mean  env_mod_rate_L_std  \
0      -0.076816           0.013925         2.510726            0.296017   
1      -0.053170          -0.011675         2.189911            0.076190   
2       0.073234           0.010955         1.217854            0.413152   
3       0.077393           0.027533         1.394646            0.467633   
4       0.022059           0.005688         1.268945            0.309523   

   env_mod_rate_R_std  mean_burst_duration_R_std  env_mod_rate_R_mean  \
0            0.312178                   0.002180             0.421821   
1            0.063898                   0.012065             0.208876   
2            0.278854                   0.001874             0.462511   
3            0.511813                   0.003089             0.637478   
4            0.341810                   0.004726             0.295681   

   PD_Label_Combined  
0                  0  
1                  0  
2      

In [3]:
# ============================================================
#     SIMPLE + STABLE DANN + SCL + MLP CLASSIFIER
#     CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function
import torch.nn.functional as F

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LOAD DATA
# ============================================================

# First 7 columns -> Features
# Last column     -> PD / HC labels


# ============================================================
# MEG FEATURES AND LABELS
# ============================================================

# All columns except the last one are MEG features
X_meg = final_meg_df.iloc[:, :-1].copy()

# Convert features to numeric
X_meg = X_meg.apply(
    pd.to_numeric,
    errors="coerce"
)

# Replace +inf and -inf with NaN
X_meg = X_meg.replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------
# Check invalid values
# ------------------------------------------------------------

print(
    "NaN values before cleaning:",
    X_meg.isna().sum().sum()
)

print(
    "Infinite values before cleaning:",
    np.isinf(
        X_meg.select_dtypes(include=np.number)
    ).sum().sum()
)

# ------------------------------------------------------------
# Replace NaN with column median
# ------------------------------------------------------------

X_meg = X_meg.fillna(
    X_meg.median(numeric_only=True)
)

# Any remaining NaN -> 0
X_meg = X_meg.fillna(0)

# Convert to float64 to avoid float32 overflow
X_meg = X_meg.values.astype(np.float64)

# Labels
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int64)

# ============================================================
# FINAL CHECK BEFORE STANDARDIZATION
# ============================================================

print(
    "All MEG values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Maximum absolute MEG value:",
    np.max(np.abs(X_meg))
)

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_meg = StandardScaler()

X_meg = scaler_meg.fit_transform(
    X_meg
)

# ============================================================
# FINAL CHECK AFTER STANDARDIZATION
# ============================================================

print(
    "All standardized values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Standardized MEG shape:",
    X_meg.shape
)

# ============================================================
# DOMAIN LABELS
# ============================================================

# 0 -> Acoustic
# 1 -> MEG

domain_meg = np.ones(
    len(X_meg)
)

# ============================================================
# COMBINE
# ============================================================

X_all = np.vstack([

    X_meg
]).astype(np.float32)

task_labels = np.hstack([

    y_meg
]).astype(np.int64)

domain_labels = np.hstack([

    domain_meg
]).astype(np.int64)

# ============================================================
# TENSORS
# ============================================================

X_all_tensor = torch.tensor(
    X_all
).float().to(device)

task_labels_tensor = torch.tensor(
    task_labels
).long().to(device)

domain_labels_tensor = torch.tensor(
    domain_labels
).long().to(device)

# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = lambd

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):

    return GradReverse.apply(x, lambd)

# ============================================================
# SIMPLE RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),
            nn.ReLU(),

            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )

# ============================================================
# DANN + SCL MODEL
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # ====================================================
        # SIMPLE ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        # ====================================================
        # TASK HEAD
        # ====================================================

        self.task_head = nn.Sequential(

            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        # ====================================================
        # DOMAIN HEAD
        # ====================================================

        self.domain_head = nn.Sequential(

            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        # L2 NORMALIZATION

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        # DANN

        rev_z = grad_reverse(
            z_norm,
            lambd
        )

        task_out = self.task_head(z_norm)

        domain_out = self.domain_head(rev_z)

        return task_out, domain_out, z_norm

# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        device = features.device

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(
            mask.shape[0]
        ).to(device)

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(1, keepdim=True) + 1e-8
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(1) / (mask.sum(1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# ============================================================
# DATA LOADER
# ============================================================

dataset = TensorDataset(
    X_all_tensor,
    task_labels_tensor,
    domain_labels_tensor
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = DANNContrastive(
    input_dim=X_all.shape[1],
    latent_dim=32
).to(device)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LOSSES
# ============================================================

task_loss_fn = nn.CrossEntropyLoss()

domain_loss_fn = nn.CrossEntropyLoss()

contrastive_loss_fn = SupConLoss(
    temperature=0.07
)

epochs = 120

print("\nTraining Started...\n")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(epochs):

    model.train()

    total_correct = 0
    total_samples = 0

    # ========================================================
    # WEAKER DANN
    # ========================================================

    p = min(epoch / 50.0, 1.0)

    lambd = 0.25 * (
        2 / (1 + np.exp(-10 * p)) - 1
    )

    for xb, y_task, y_dom in loader:

        optimizer.zero_grad()

        task_out, dom_out, z = model(
            xb,
            lambd
        )

        # ====================================================
        # LOSSES
        # ====================================================

        loss_task = task_loss_fn(
            task_out,
            y_task
        )

        loss_domain = domain_loss_fn(
            dom_out,
            y_dom
        )

        loss_contrastive = contrastive_loss_fn(
            z,
            y_task
        )

        # ====================================================
        # FINAL LOSS
        # ====================================================

        loss = (
            3.0 * loss_task +
            0.25 * loss_domain +
            0.55 * loss_contrastive
        )

        loss.backward()

        optimizer.step()

        preds = task_out.argmax(1)

        total_correct += (
            preds == y_task
        ).sum().item()

        total_samples += len(y_task)

    acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| λ={lambd:.3f} "
        f"| Acc={acc:.4f}"
    )

# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

model.eval()

with torch.no_grad():

    fused_meg_32D = model(
        torch.tensor(X_meg).float().to(device),
        0
    )[2].cpu().numpy()

# ============================================================
# COMBINE LATENTS
# ============================================================

X_fused = np.vstack([

    fused_meg_32D
])

y_fused = np.hstack([

    y_meg
])

# ============================================================
# SIMPLE MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# CROSS PARTICIPANT EVALUATION
# ============================================================

print("\n======================================")
print(" CROSS PARTICIPANT ANALYSIS")
print("======================================\n")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accs = []
precs = []
recs = []
f1s = []
aucs = []

fold = 1

for train_idx, test_idx in skf.split(X_fused, y_fused):

    X_train = torch.tensor(
        X_fused[train_idx]
    ).float().to(device)

    X_test = torch.tensor(
        X_fused[test_idx]
    ).float().to(device)

    y_train = torch.tensor(
        y_fused[train_idx]
    ).long().to(device)

    y_test = y_fused[test_idx]

    clf = FinalMLP().to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # TRAIN MLP
    # ========================================================

    clf.train()

    for epoch in range(80):

        optimizer_clf.zero_grad()

        outputs = clf(X_train)

        loss = criterion(
            outputs,
            y_train
        )

        loss.backward()

        optimizer_clf.step()

    # ========================================================
    # TEST
    # ========================================================

    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, preds)

    prec = precision_score(y_test, preds)

    rec = recall_score(y_test, preds)

    f1 = f1_score(y_test, preds)

    auc = roc_auc_score(y_test, probs)

    accs.append(acc)
    precs.append(prec)
    recs.append(rec)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold}")

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    print()

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print(" FINAL RESULTS")
print("======================================\n")

print(f"Mean Accuracy  : {np.mean(accs)*100:.2f}%")
print(f"Mean Precision : {np.mean(precs)*100:.2f}%")
print(f"Mean Recall    : {np.mean(recs)*100:.2f}%")
print(f"Mean F1-Score  : {np.mean(f1s)*100:.2f}%")
print(f"Mean AUC       : {np.mean(aucs)*100:.2f}%")

print("\n======================================")
print(" LATENT SHAPES")
print("======================================\n")


print("MEG      :", fused_meg_32D.shape)

NaN values before cleaning: 0
Infinite values before cleaning: 0
All MEG values finite: True
Maximum absolute MEG value: 12400.918555662023
All standardized values finite: True
Standardized MEG shape: (127, 7)

Training Started...

Epoch 1/120 | λ=0.000 | Acc=0.4882
Epoch 2/120 | λ=0.025 | Acc=0.4882
Epoch 3/120 | λ=0.049 | Acc=0.4882
Epoch 4/120 | λ=0.073 | Acc=0.4882
Epoch 5/120 | λ=0.095 | Acc=0.4882
Epoch 6/120 | λ=0.116 | Acc=0.4882
Epoch 7/120 | λ=0.134 | Acc=0.5118
Epoch 8/120 | λ=0.151 | Acc=0.4882
Epoch 9/120 | λ=0.166 | Acc=0.4724
Epoch 10/120 | λ=0.179 | Acc=0.4803
Epoch 11/120 | λ=0.190 | Acc=0.4646
Epoch 12/120 | λ=0.200 | Acc=0.5512
Epoch 13/120 | λ=0.208 | Acc=0.6063
Epoch 14/120 | λ=0.215 | Acc=0.5984
Epoch 15/120 | λ=0.221 | Acc=0.5748
Epoch 16/120 | λ=0.226 | Acc=0.6457
Epoch 17/120 | λ=0.230 | Acc=0.5591
Epoch 18/120 | λ=0.234 | Acc=0.5827
Epoch 19/120 | λ=0.237 | Acc=0.6142
Epoch 20/120 | λ=0.239 | Acc=0.5433
Epoch 21/120 | λ=0.241 | Acc=0.5433
Epoch 22/120 | λ=0.24

In [4]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)

fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([127, 32])
torch.float32


In [5]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)


# MEG target from final_meg_df (last column)
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
 # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

# (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target

meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([127, 32]), dtype: torch.float32
Labels shape: torch.Size([127]), dtype: torch.int64


In [6]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(127, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0 -0.239895 -0.097552  0.031369  0.140410 -0.007474  0.078755 -0.108125   
1 -0.195719 -0.090690  0.053638  0.199227  0.021527  0.076620 -0.091155   
2 -0.185554 -0.122462  0.001579  0.124837  0.069647  0.050060 -0.072836   
3  0.197814 -0.219312 -0.256816 -0.209268  0.247721 -0.133754  0.244967   
4  0.161695 -0.247743 -0.295114 -0.210241  0.253852 -0.100092  0.255882   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.109131  0.208592 -0.140864  ...  0.076678 -0.271674  0.252530  0.048687   
1  0.096063  0.179910 -0.226530  ...  0.073039 -0.306982  0.194017  0.089196   
2  0.069081  0.207931 -0.188298  ...  0.115876 -0.342871  0.218073  0.058654   
3 -0.188240  0.237691  0.138394  ...  0.287918 -0.255241 -0.011113  0.068343   
4 -0.175399  0.207868  0.168286  ...  0.273344 -0.249890  0.028523  0.076685   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [7]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd

fused_df_named_int.to_csv('/content/drive//MyDrive/Modality-top7_meg(passive)_par_MIRA-Net_data.csv', index=False)

In [10]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int[
    [f'feat_{i}' for i in range(32)]
].copy()

y = fused_df_named_int[
    "PD_Label_Combined"
].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 32)
Test Shape  : (26, 32)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 1.0000
Precision           : 1.0000
Sensitivity         : 1.0000
F1 Score            : 1.0000
ROC-AUC             : 1.0000
Balanced Accuracy   : 1.0000

XGBoost | Fold 2/10
Accuracy            : 1.0000
Precision           : 1.0000
Sensitivity         : 1.0000
F1 Score            : 1.0000
ROC-AUC            

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
XGBoost,0.98,1.0,0.96,0.975000,0.998,0.98,1.0,0.871271,1.0,1.0,...,0.771905,1.0,1.0,1.0,1.0,1.0,13.0,0.0,0.0,13.0
Random Forest,0.99,1.0,0.98,0.988889,0.998,0.99,1.0,0.871271,1.0,1.0,...,0.771905,1.0,1.0,1.0,1.0,1.0,13.0,0.0,0.0,13.0
HistGradientBoosting,0.99,1.0,0.98,0.988889,0.990,0.99,1.0,0.871271,1.0,1.0,...,0.771905,1.0,1.0,1.0,1.0,1.0,13.0,0.0,0.0,13.0
Voting Fusion,0.99,1.0,0.98,0.988889,0.998,0.99,1.0,0.871271,1.0,1.0,...,0.771905,1.0,1.0,1.0,1.0,1.0,13.0,0.0,0.0,13.0
Stacking Fusion,0.99,1.0,0.98,0.988889,0.998,0.99,1.0,0.871271,1.0,1.0,...,0.771905,1.0,1.0,1.0,1.0,1.0,13.0,0.0,0.0,13.0


Without MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-passive(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   env_kurt_asym  burst_amp_cv_asym  env_kurt_R_mean  env_mod_rate_L_std  \
0      -0.076816           0.013925         2.510726            0.296017   
1      -0.053170          -0.011675         2.189911            0.076190   
2       0.073234           0.010955         1.217854            0.413152   
3       0.077393           0.027533         1.394646            0.467633   
4       0.022059           0.005688         1.268945            0.309523   

   env_mod_rate_R_std  mean_burst_duration_R_std  env_mod_rate_R_mean  \
0            0.312178                   0.002180             0.421821   
1            0.063898                   0.012065             0.208876   
2            0.278854                   0.001874             0.462511   
3            0.511813                   0.003089             0.637478   
4            0.341810                   0.004726             0.295681   

   PD_Label_Combined  
0                  0  
1                  0  
2      

In [3]:
fused_df_named_int = final_meg_df.rename(columns={"Pd_indicator_meg": "PD_Label_Combined"})
fused_df_named_int.head()
fused_df_named_int.shape

(127, 8)

In [4]:
fused_df_named_int.head()

,env_kurt_asym,burst_amp_cv_asym,env_kurt_R_mean,env_mod_rate_L_std,env_mod_rate_R_std,mean_burst_duration_R_std,env_mod_rate_R_mean,PD_Label_Combined
0,-0.076816,0.013925,2.510726,0.296017,0.312178,0.002180,0.421821,0
1,-0.053170,-0.011675,2.189911,0.076190,0.063898,0.012065,0.208876,0
2,0.073234,0.010955,1.217854,0.413152,0.278854,0.001874,0.462511,0
3,0.077393,0.027533,1.394646,0.467633,0.511813,0.003089,0.637478,1
4,0.022059,0.005688,1.268945,0.309523,0.341810,0.004726,0.295681,1


In [5]:
# ============================================================
# ###8)###
#
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

# All feature columns except the final label column
X = fused_df_named_int.iloc[:, :-1].copy()

# Target
y = fused_df_named_int["PD_Label_Combined"].copy()


# ============================================================
# CONVERT FEATURES TO NUMERIC
# ============================================================

X = X.apply(
    pd.to_numeric,
    errors="coerce"
)


# ============================================================
# CHECK RAW VALUES
# ============================================================

print("\nRaw feature statistics:")
print("Shape :", X.shape)

print(
    "NaN values before cleaning :",
    X.isna().sum().sum()
)

print(
    "Inf values before cleaning :",
    np.isinf(
        X.to_numpy(dtype=np.float64)
    ).sum()
)


# ============================================================
# HANDLE +INF / -INF
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# COLUMN-WISE MEDIAN IMPUTATION
# ============================================================

col_medians = X.median(
    axis=0,
    skipna=True
)

# If an entire column is NaN,
# replace its median with 0
col_medians = col_medians.fillna(0)

X = X.fillna(
    col_medians
)


# ============================================================
# CLIP EXTREMELY LARGE VALUES
# ============================================================

# Prevent overflow when converting to float32.
#
# Values larger than this are clipped rather than allowed
# to produce inf during float32 conversion.

CLIP_VALUE = 1e10

X = X.clip(
    lower=-CLIP_VALUE,
    upper=CLIP_VALUE
)


# ============================================================
# CONVERT TO FLOAT32 AFTER CLEANING
# ============================================================

X = X.astype(
    np.float32
)


# ============================================================
# FINAL FINITE-VALUE CHECK
# ============================================================

X_array = X.to_numpy()

print("\nAfter cleaning:")

print(
    "NaN values :",
    np.isnan(X_array).sum()
)

print(
    "Inf values :",
    np.isinf(X_array).sum()
)

print(
    "Finite     :",
    np.isfinite(X_array).all()
)

print(
    "Maximum    :",
    np.max(X_array)
)

print(
    "Minimum    :",
    np.min(X_array)
)


# ============================================================
# TARGET CLEANING
# ============================================================

y = pd.to_numeric(
    y,
    errors="coerce"
)

# Make sure target contains no missing values
valid_idx = y.notna()

X = X.loc[valid_idx].reset_index(drop=True)

y = y.loc[valid_idx].astype(
    np.int64
).reset_index(drop=True)


print("\nFinal data:")
print("X shape :", X.shape)
print("y shape :", y.shape)

print("\nClass distribution:")
print(y.value_counts())


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)


print("\nTraining Class Distribution:")
print(
    y_train.value_counts()
)


print("\nTesting Class Distribution:")
print(
    y_test.value_counts()
)


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw_dict[1] / cw_dict[0]
)


print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)




# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        scaler = StandardScaler()
        X_tr = X_tr.astype(np.float64)
        X_val = X_val.astype(np.float64)
        X_tr = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        # ----------------------------------------------------
        # Safety check after scaling
        # ----------------------------------------------------
        X_tr_scaled = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        print(
            f"Fold {fold}: "
            f"max|X_val|={np.max(np.abs(X_val_scaled)):.4f}"
        )

        val_preds = model.predict(
             X_val_scaled
        )
        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]

        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00

Raw feature statistics:
Shape : (127, 7)
NaN values before cleaning : 0
Inf values before cleaning : 0

After cleaning:
NaN values : 0
Inf values : 0
Finite     : True
Maximum    : 12400.919
Minimum    : -0.55352616

Final data:
X shape : (127, 7)
y shape : (127,)

Class distribution:
PD_Label_Combined
0    65
1    62
Name: count, dtype: int64

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 7)
Test Shape  : (26, 7)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoo

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
Random Forest,0.703636,0.726429,0.67,0.667778,0.7805,0.703333,0.692308,0.500117,0.834987,0.666667,...,0.355229,0.822903,0.714286,0.736686,0.692308,0.389249,8.0,5.0,3.0,10.0
Stacking Fusion,0.634545,0.626667,0.61,0.607359,0.7215,0.636667,0.692308,0.500117,0.834987,0.692308,...,0.423693,0.873193,0.692308,0.704142,0.692308,0.384615,9.0,4.0,4.0,9.0
HistGradientBoosting,0.663636,0.708333,0.62,0.637900,0.7110,0.665000,0.653846,0.462206,0.805878,0.642857,...,0.355229,0.822903,0.666667,0.686391,0.653846,0.308607,8.0,5.0,4.0,9.0
Voting Fusion,0.693636,0.740000,0.66,0.662677,0.7750,0.696667,0.653846,0.462206,0.805878,0.642857,...,0.355229,0.822903,0.666667,0.662722,0.653846,0.308607,8.0,5.0,4.0,9.0
XGBoost,0.714545,0.747143,0.70,0.686465,0.7910,0.720000,0.615385,0.425349,0.775714,0.615385,...,0.355229,0.822903,0.615385,0.644970,0.615385,0.230769,8.0,5.0,5.0,8.0


***Meg Rest task***

With MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-rest(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   ibi_cv_asym  ibi_cv_L_std  mean_burst_duration_R_std  \
0     0.002893      0.048081                   0.002407   
1    -0.018062      0.085531                   0.005160   
2    -0.002573      0.032173                   0.001270   
3     0.001531      0.044880                   0.003106   
4     0.023834      0.142267                   0.004734   

   mean_burst_duration_asym  burst_dur_cv_R_std  burst_dur_cv_L_std  \
0                  0.009156            0.040693            0.055777   
1                 -0.005581            0.048648            0.052883   
2                  0.007325            0.030995            0.046488   
3                  0.005372            0.042429            0.049082   
4                  0.014897            0.044595            0.053040   

   ibi_cv_R_std  PD_Label_Combined  
0      0.043907                  0  
1      0.086747                  0  
2      0.032718                  0  
3      0.044973                  1  
4    

In [3]:
# ============================================================
#     SIMPLE + STABLE DANN + SCL + MLP CLASSIFIER
#     CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function
import torch.nn.functional as F

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LOAD DATA
# ============================================================

# First 7 columns -> Features
# Last column     -> PD / HC labels


# ============================================================
# MEG FEATURES AND LABELS
# ============================================================

# All columns except the last one are MEG features
X_meg = final_meg_df.iloc[:, :-1].copy()

# Convert features to numeric
X_meg = X_meg.apply(
    pd.to_numeric,
    errors="coerce"
)

# Replace +inf and -inf with NaN
X_meg = X_meg.replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------
# Check invalid values
# ------------------------------------------------------------

print(
    "NaN values before cleaning:",
    X_meg.isna().sum().sum()
)

print(
    "Infinite values before cleaning:",
    np.isinf(
        X_meg.select_dtypes(include=np.number)
    ).sum().sum()
)

# ------------------------------------------------------------
# Replace NaN with column median
# ------------------------------------------------------------

X_meg = X_meg.fillna(
    X_meg.median(numeric_only=True)
)

# Any remaining NaN -> 0
X_meg = X_meg.fillna(0)

# Convert to float64 to avoid float32 overflow
X_meg = X_meg.values.astype(np.float64)

# Labels
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int64)

# ============================================================
# FINAL CHECK BEFORE STANDARDIZATION
# ============================================================

print(
    "All MEG values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Maximum absolute MEG value:",
    np.max(np.abs(X_meg))
)

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_meg = StandardScaler()

X_meg = scaler_meg.fit_transform(
    X_meg
)

# ============================================================
# FINAL CHECK AFTER STANDARDIZATION
# ============================================================

print(
    "All standardized values finite:",
    np.isfinite(X_meg).all()
)

print(
    "Standardized MEG shape:",
    X_meg.shape
)

# ============================================================
# DOMAIN LABELS
# ============================================================

# 0 -> Acoustic
# 1 -> MEG

domain_meg = np.ones(
    len(X_meg)
)

# ============================================================
# COMBINE
# ============================================================

X_all = np.vstack([

    X_meg
]).astype(np.float32)

task_labels = np.hstack([

    y_meg
]).astype(np.int64)

domain_labels = np.hstack([

    domain_meg
]).astype(np.int64)

# ============================================================
# TENSORS
# ============================================================

X_all_tensor = torch.tensor(
    X_all
).float().to(device)

task_labels_tensor = torch.tensor(
    task_labels
).long().to(device)

domain_labels_tensor = torch.tensor(
    domain_labels
).long().to(device)

# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = lambd

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):

    return GradReverse.apply(x, lambd)

# ============================================================
# SIMPLE RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),
            nn.ReLU(),

            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )

# ============================================================
# DANN + SCL MODEL
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # ====================================================
        # SIMPLE ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        # ====================================================
        # TASK HEAD
        # ====================================================

        self.task_head = nn.Sequential(

            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        # ====================================================
        # DOMAIN HEAD
        # ====================================================

        self.domain_head = nn.Sequential(

            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        # L2 NORMALIZATION

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        # DANN

        rev_z = grad_reverse(
            z_norm,
            lambd
        )

        task_out = self.task_head(z_norm)

        domain_out = self.domain_head(rev_z)

        return task_out, domain_out, z_norm

# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        device = features.device

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(
            mask.shape[0]
        ).to(device)

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(1, keepdim=True) + 1e-8
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(1) / (mask.sum(1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# ============================================================
# DATA LOADER
# ============================================================

dataset = TensorDataset(
    X_all_tensor,
    task_labels_tensor,
    domain_labels_tensor
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = DANNContrastive(
    input_dim=X_all.shape[1],
    latent_dim=32
).to(device)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LOSSES
# ============================================================

task_loss_fn = nn.CrossEntropyLoss()

domain_loss_fn = nn.CrossEntropyLoss()

contrastive_loss_fn = SupConLoss(
    temperature=0.07
)

epochs = 120

print("\nTraining Started...\n")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(epochs):

    model.train()

    total_correct = 0
    total_samples = 0

    # ========================================================
    # WEAKER DANN
    # ========================================================

    p = min(epoch / 50.0, 1.0)

    lambd = 0.25 * (
        2 / (1 + np.exp(-10 * p)) - 1
    )

    for xb, y_task, y_dom in loader:

        optimizer.zero_grad()

        task_out, dom_out, z = model(
            xb,
            lambd
        )

        # ====================================================
        # LOSSES
        # ====================================================

        loss_task = task_loss_fn(
            task_out,
            y_task
        )

        loss_domain = domain_loss_fn(
            dom_out,
            y_dom
        )

        loss_contrastive = contrastive_loss_fn(
            z,
            y_task
        )

        # ====================================================
        # FINAL LOSS
        # ====================================================

        loss = (
            3.0 * loss_task +
            0.25 * loss_domain +
            0.55 * loss_contrastive
        )

        loss.backward()

        optimizer.step()

        preds = task_out.argmax(1)

        total_correct += (
            preds == y_task
        ).sum().item()

        total_samples += len(y_task)

    acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| λ={lambd:.3f} "
        f"| Acc={acc:.4f}"
    )

# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

model.eval()

with torch.no_grad():

    fused_meg_32D = model(
        torch.tensor(X_meg).float().to(device),
        0
    )[2].cpu().numpy()

# ============================================================
# COMBINE LATENTS
# ============================================================

X_fused = np.vstack([

    fused_meg_32D
])

y_fused = np.hstack([

    y_meg
])

# ============================================================
# SIMPLE MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# CROSS PARTICIPANT EVALUATION
# ============================================================

print("\n======================================")
print(" CROSS PARTICIPANT ANALYSIS")
print("======================================\n")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accs = []
precs = []
recs = []
f1s = []
aucs = []

fold = 1

for train_idx, test_idx in skf.split(X_fused, y_fused):

    X_train = torch.tensor(
        X_fused[train_idx]
    ).float().to(device)

    X_test = torch.tensor(
        X_fused[test_idx]
    ).float().to(device)

    y_train = torch.tensor(
        y_fused[train_idx]
    ).long().to(device)

    y_test = y_fused[test_idx]

    clf = FinalMLP().to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # TRAIN MLP
    # ========================================================

    clf.train()

    for epoch in range(80):

        optimizer_clf.zero_grad()

        outputs = clf(X_train)

        loss = criterion(
            outputs,
            y_train
        )

        loss.backward()

        optimizer_clf.step()

    # ========================================================
    # TEST
    # ========================================================

    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, preds)

    prec = precision_score(y_test, preds)

    rec = recall_score(y_test, preds)

    f1 = f1_score(y_test, preds)

    auc = roc_auc_score(y_test, probs)

    accs.append(acc)
    precs.append(prec)
    recs.append(rec)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold}")

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    print()

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print(" FINAL RESULTS")
print("======================================\n")

print(f"Mean Accuracy  : {np.mean(accs)*100:.2f}%")
print(f"Mean Precision : {np.mean(precs)*100:.2f}%")
print(f"Mean Recall    : {np.mean(recs)*100:.2f}%")
print(f"Mean F1-Score  : {np.mean(f1s)*100:.2f}%")
print(f"Mean AUC       : {np.mean(aucs)*100:.2f}%")

print("\n======================================")
print(" LATENT SHAPES")
print("======================================\n")


print("MEG      :", fused_meg_32D.shape)

NaN values before cleaning: 0
Infinite values before cleaning: 0
All MEG values finite: True
Maximum absolute MEG value: 9.720063404245048
All standardized values finite: True
Standardized MEG shape: (127, 7)

Training Started...

Epoch 1/120 | λ=0.000 | Acc=0.4882
Epoch 2/120 | λ=0.025 | Acc=0.4961
Epoch 3/120 | λ=0.049 | Acc=0.4882
Epoch 4/120 | λ=0.073 | Acc=0.4567
Epoch 5/120 | λ=0.095 | Acc=0.4882
Epoch 6/120 | λ=0.116 | Acc=0.4882
Epoch 7/120 | λ=0.134 | Acc=0.4724
Epoch 8/120 | λ=0.151 | Acc=0.5276
Epoch 9/120 | λ=0.166 | Acc=0.5433
Epoch 10/120 | λ=0.179 | Acc=0.5512
Epoch 11/120 | λ=0.190 | Acc=0.4961
Epoch 12/120 | λ=0.200 | Acc=0.4961
Epoch 13/120 | λ=0.208 | Acc=0.5512
Epoch 14/120 | λ=0.215 | Acc=0.5276
Epoch 15/120 | λ=0.221 | Acc=0.4646
Epoch 16/120 | λ=0.226 | Acc=0.5433
Epoch 17/120 | λ=0.230 | Acc=0.5433
Epoch 18/120 | λ=0.234 | Acc=0.5118
Epoch 19/120 | λ=0.237 | Acc=0.5748
Epoch 20/120 | λ=0.239 | Acc=0.5197
Epoch 21/120 | λ=0.241 | Acc=0.5748
Epoch 22/120 | λ=0.243

In [4]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)

fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([127, 32])
torch.float32


In [5]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)


# MEG target from final_meg_df (last column)
y_meg = final_meg_df.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
 # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

# (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target

meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([127, 32]), dtype: torch.float32
Labels shape: torch.Size([127]), dtype: torch.int64


In [6]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(127, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0 -0.205202  0.202155  0.395917 -0.142079 -0.251269  0.154868  0.093341   
1 -0.192544  0.196861  0.375554 -0.196944 -0.220140  0.156829  0.086638   
2 -0.181734  0.200657  0.393347 -0.156078 -0.212128  0.178717  0.093244   
3 -0.168545  0.083681  0.323453 -0.109676 -0.356344  0.015953  0.085786   
4 -0.129973  0.088164  0.413161 -0.088303 -0.354400  0.035773  0.055469   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.147236  0.017109  0.187730  ...  0.044734 -0.036165  0.132170  0.162390   
1  0.177140  0.012826  0.172384  ...  0.062142  0.021189  0.155198  0.113645   
2  0.176352 -0.025954  0.184620  ...  0.074627 -0.006283  0.136533  0.121249   
3  0.022753  0.118257  0.134121  ... -0.095680 -0.196230  0.066993  0.277138   
4  0.052083  0.137480  0.128332  ... -0.105679 -0.180321  0.057451  0.253116   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [7]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd

fused_df_named_int.to_csv('/content/drive//MyDrive/Modality-top7_meg(rest)_par_MIRA-Net_data.csv', index=False)

In [10]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int[
    [f'feat_{i}' for i in range(32)]
].copy()

y = fused_df_named_int[
    "PD_Label_Combined"
].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 32)
Test Shape  : (26, 32)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 1.0000
Precision           : 1.0000
Sensitivity         : 1.0000
F1 Score            : 1.0000
ROC-AUC             : 1.0000
Balanced Accuracy   : 1.0000

XGBoost | Fold 2/10
Accuracy            : 0.8000
Precision           : 0.6667
Sensitivity         : 1.0000
F1 Score            : 0.8000
ROC-AUC            

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
HistGradientBoosting,0.930000,0.957143,0.92,0.925505,0.985667,0.935000,0.923077,0.758584,0.978645,0.923077,...,0.666860,0.986290,0.923077,0.994083,0.923077,0.846154,12.0,1.0,1.0,12.0
Voting Fusion,0.950000,0.957143,0.96,0.950505,0.989833,0.955000,0.923077,0.758584,0.978645,0.923077,...,0.666860,0.986290,0.923077,0.988166,0.923077,0.846154,12.0,1.0,1.0,12.0
Stacking Fusion,0.930909,0.940476,0.94,0.930303,0.998000,0.936667,0.884615,0.710241,0.959968,0.812500,...,0.497436,0.918205,0.896552,0.982249,0.884615,0.790569,10.0,3.0,0.0,13.0
XGBoost,0.940000,0.966667,0.92,0.930000,0.989833,0.943333,0.884615,0.710241,0.959968,0.857143,...,0.577654,0.956742,0.888889,0.982249,0.884615,0.771517,11.0,2.0,1.0,12.0
Random Forest,0.930909,0.940476,0.94,0.927525,0.975583,0.936667,0.884615,0.710241,0.959968,0.857143,...,0.577654,0.956742,0.888889,0.982249,0.884615,0.771517,11.0,2.0,1.0,12.0


Without MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

final_meg_df = pd.read_csv('/content/drive/MyDrive/final_meg-rest(par)_top7_rfc+shap_10-fold_analysis_last_column.csv')

print("Loaded data shape:", final_meg_df.shape)
print(final_meg_df.head())


Loaded data shape: (127, 8)
   ibi_cv_asym  ibi_cv_L_std  mean_burst_duration_R_std  \
0     0.002893      0.048081                   0.002407   
1    -0.018062      0.085531                   0.005160   
2    -0.002573      0.032173                   0.001270   
3     0.001531      0.044880                   0.003106   
4     0.023834      0.142267                   0.004734   

   mean_burst_duration_asym  burst_dur_cv_R_std  burst_dur_cv_L_std  \
0                  0.009156            0.040693            0.055777   
1                 -0.005581            0.048648            0.052883   
2                  0.007325            0.030995            0.046488   
3                  0.005372            0.042429            0.049082   
4                  0.014897            0.044595            0.053040   

   ibi_cv_R_std  PD_Label_Combined  
0      0.043907                  0  
1      0.086747                  0  
2      0.032718                  0  
3      0.044973                  1  
4    

In [3]:
fused_df_named_int = final_meg_df.rename(columns={"Pd_indicator_meg": "PD_Label_Combined"})
fused_df_named_int.head()
fused_df_named_int.shape

(127, 8)

In [4]:
fused_df_named_int.head()

,ibi_cv_asym,ibi_cv_L_std,mean_burst_duration_R_std,mean_burst_duration_asym,burst_dur_cv_R_std,burst_dur_cv_L_std,ibi_cv_R_std,PD_Label_Combined
0,0.002893,0.048081,0.002407,0.009156,0.040693,0.055777,0.043907,0
1,-0.018062,0.085531,0.005160,-0.005581,0.048648,0.052883,0.086747,0
2,-0.002573,0.032173,0.001270,0.007325,0.030995,0.046488,0.032718,0
3,0.001531,0.044880,0.003106,0.005372,0.042429,0.049082,0.044973,1
4,0.023834,0.142267,0.004734,0.014897,0.044595,0.053040,0.113787,1


In [5]:
# ============================================================
# ###8)###
#
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

# All feature columns except the final label column
X = fused_df_named_int.iloc[:, :-1].copy()

# Target
y = fused_df_named_int["PD_Label_Combined"].copy()


# ============================================================
# CONVERT FEATURES TO NUMERIC
# ============================================================

X = X.apply(
    pd.to_numeric,
    errors="coerce"
)


# ============================================================
# CHECK RAW VALUES
# ============================================================

print("\nRaw feature statistics:")
print("Shape :", X.shape)

print(
    "NaN values before cleaning :",
    X.isna().sum().sum()
)

print(
    "Inf values before cleaning :",
    np.isinf(
        X.to_numpy(dtype=np.float64)
    ).sum()
)


# ============================================================
# HANDLE +INF / -INF
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# COLUMN-WISE MEDIAN IMPUTATION
# ============================================================

col_medians = X.median(
    axis=0,
    skipna=True
)

# If an entire column is NaN,
# replace its median with 0
col_medians = col_medians.fillna(0)

X = X.fillna(
    col_medians
)


# ============================================================
# CLIP EXTREMELY LARGE VALUES
# ============================================================

# Prevent overflow when converting to float32.
#
# Values larger than this are clipped rather than allowed
# to produce inf during float32 conversion.

CLIP_VALUE = 1e10

X = X.clip(
    lower=-CLIP_VALUE,
    upper=CLIP_VALUE
)


# ============================================================
# CONVERT TO FLOAT32 AFTER CLEANING
# ============================================================

X = X.astype(
    np.float32
)


# ============================================================
# FINAL FINITE-VALUE CHECK
# ============================================================

X_array = X.to_numpy()

print("\nAfter cleaning:")

print(
    "NaN values :",
    np.isnan(X_array).sum()
)

print(
    "Inf values :",
    np.isinf(X_array).sum()
)

print(
    "Finite     :",
    np.isfinite(X_array).all()
)

print(
    "Maximum    :",
    np.max(X_array)
)

print(
    "Minimum    :",
    np.min(X_array)
)


# ============================================================
# TARGET CLEANING
# ============================================================

y = pd.to_numeric(
    y,
    errors="coerce"
)

# Make sure target contains no missing values
valid_idx = y.notna()

X = X.loc[valid_idx].reset_index(drop=True)

y = y.loc[valid_idx].astype(
    np.int64
).reset_index(drop=True)


print("\nFinal data:")
print("X shape :", X.shape)
print("y shape :", y.shape)

print("\nClass distribution:")
print(y.value_counts())


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)


print("\nTraining Class Distribution:")
print(
    y_train.value_counts()
)


print("\nTesting Class Distribution:")
print(
    y_test.value_counts()
)


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw_dict[1] / cw_dict[0]
)


print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)




# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        scaler = StandardScaler()
        X_tr = X_tr.astype(np.float64)
        X_val = X_val.astype(np.float64)
        X_tr = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        # ----------------------------------------------------
        # Safety check after scaling
        # ----------------------------------------------------
        X_tr_scaled = np.nan_to_num(
            X_tr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        X_val = np.nan_to_num(
            X_val,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        print(
            f"Fold {fold}: "
            f"max|X_val|={np.max(np.abs(X_val_scaled)):.4f}"
        )

        val_preds = model.predict(
             X_val_scaled
        )
        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]

        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00

Raw feature statistics:
Shape : (127, 7)
NaN values before cleaning : 0
Inf values before cleaning : 0

After cleaning:
NaN values : 0
Inf values : 0
Finite     : True
Maximum    : 9.720063
Minimum    : -0.57448953

Final data:
X shape : (127, 7)
y shape : (127,)

Class distribution:
PD_Label_Combined
0    65
1    62
Name: count, dtype: int64

Class Weights:
{0: 0.9769230769230769, 1: 1.0241935483870968}

Train Shape : (101, 7)
Test Shape  : (26, 7)

Training Class Distribution:
PD_Label_Combined
0    52
1    49
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    13
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(0.9711538461538461), np.int64(1): np.float64(1.030612244897959)}

XGBoost scale_pos_weight: 1.0612244897959182


################################################
Evaluating: XGBoost
################################################

XGBoos

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
HistGradientBoosting,0.591818,0.608095,0.590,0.566566,0.607833,0.595000,0.807692,0.621237,0.914929,0.785714,...,0.497436,0.918205,0.814815,0.869822,0.807692,0.617213,10.0,3.0,2.0,11.0
XGBoost,0.542727,0.517143,0.555,0.518521,0.570833,0.545833,0.769231,0.579484,0.889662,0.769231,...,0.497436,0.918205,0.769231,0.863905,0.769231,0.538462,10.0,3.0,3.0,10.0
Random Forest,0.591818,0.593095,0.615,0.581082,0.606500,0.597500,0.769231,0.579484,0.889662,0.769231,...,0.497436,0.918205,0.769231,0.849112,0.769231,0.538462,10.0,3.0,3.0,10.0
Voting Fusion,0.571818,0.547619,0.555,0.531349,0.577500,0.577500,0.730769,0.539170,0.862956,0.714286,...,0.423693,0.873193,0.740741,0.863905,0.730769,0.462910,9.0,4.0,3.0,10.0
Stacking Fusion,0.565455,0.550000,0.590,0.560528,0.588333,0.563333,0.730769,0.539170,0.862956,0.750000,...,0.497436,0.918205,0.720000,0.781065,0.730769,0.462910,10.0,3.0,4.0,9.0
